In [ ]:
import cv2
import csv
import os
from ultralytics import YOLO

VIDEO_PATH = "videos/sample_video.mp4"  
OUTPUT_PATH = "outputs/humans_out.mp4"
STATS_PATH = "outputs/humans_stats.csv"
VIEW = True  

In [ ]:
class HumanDetector:
    def __init__(self, model_name="models/yolov8m.pt", conf=0.5, iou=0.5, device=None):
        self.model = YOLO(model_name)
        self.conf = conf
        self.iou = iou
        self.device = device

    def infer_frame(self, frame):
        results = self.model.predict(
            source=frame,
            conf=self.conf,
            iou=self.iou,
            device=self.device,
            classes=[0],
            verbose=False
        )
        dets = []
        for r in results:
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xyxy = r.boxes.xyxy.cpu().numpy()
            for (x1, y1, x2, y2) in xyxy:
                dets.append((int(x1), int(y1), int(x2), int(y2)))
        return dets

In [ ]:
def draw_box(frame, bbox, label="Human"):
    x1, y1, x2, y2 = bbox
    color = (0, 255, 0)
    thickness = 2
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    cv2.putText(frame, label, (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

In [ ]:
def run_video(video_path, output_path=None, stats_path=None, view=True):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open {video_path}")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h)) if output_path else None

    detector = HumanDetector()
    stats = []

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        dets = detector.infer_frame(frame)
        count = len(dets)

        for bbox in dets:
            draw_box(frame, bbox)

        cv2.putText(frame, f"People: {count}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        stats.append([frame_idx, count])

        if view:
            cv2.imshow("Humans Detection", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        if out:
            out.write(frame)

        if frame_idx % 30 == 0:
            print(f"[INFO] Processed frame {frame_idx}, detected {count} humans.")

    cap.release()
    if out:
        out.release()
    cv2.destroyAllWindows()

    if stats_path:
        os.makedirs(os.path.dirname(stats_path), exist_ok=True)
        with open(stats_path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Frame", "Human_Count"])
            writer.writerows(stats)
        print(f"[OK] Stats saved to {stats_path}")

    print("[DONE] Human detection completed.")

In [ ]:
!python human_detection/human_detector.py --source "videos/sample3.mp4" --out "outputs/human_out.mp4" --view

In [ ]:
import cv2
from ultralytics import YOLO
import os
import csv

In [ ]:
video_path = "videos/sample1.mp4"
output_video = "outputs/mask_out.mp4"
model_path = "models/mask_yolov8n.pt"
view = True
csv_path = "outputs/mask_stats.csv"
confidence = 0.5

In [ ]:
class MaskDetector:
    def __init__(self, model_path=model_path):
        self.model = YOLO(model_path)

    def detect(self, frame, conf_threshold=confidence):
        results = self.model(frame)
        detections = []
        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                label = self.model.names[cls]
                conf = float(box.conf[0])
                if conf < conf_threshold:
                    continue
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                detections.append({
                    "label": label,
                    "confidence": conf,
                    "bbox": (int(x1), int(y1), int(x2), int(y2))
                })
        return detections


In [ ]:
def draw_box(frame, bbox, label):
    x1, y1, x2, y2 = map(int, bbox)
    if "no" in label.lower():
        color = (0, 0, 255)
        text = "No Mask"
    else:
        color = (0, 255, 0)
        text = "Mask"
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)


In [ ]:
def run_video(video_path, output_video=output_video, csv_file=csv_path, view=view):
    cap = cv2.VideoCapture(video_path)
    detector = MaskDetector(model_path)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    os.makedirs(os.path.dirname(output_video), exist_ok=True)
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    os.makedirs(os.path.dirname(csv_file), exist_ok=True)
    csvf = open(csv_file, mode='w', newline='')
    writer = csv.writer(csvf)
    writer.writerow(["Frame", "Total_Faces", "Mask", "No_Mask", "Mask_Percentage"])

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        detections = detector.detect(frame)
        mask_count = sum(1 for d in detections if "no" not in d["label"].lower())
        no_mask_count = sum(1 for d in detections if "no" in d["label"].lower())
        total_faces = mask_count + no_mask_count
        mask_percentage = (mask_count / total_faces * 100) if total_faces > 0 else 0

        writer.writerow([frame_idx, total_faces, mask_count, no_mask_count, f"{mask_percentage:.2f}"])

        for det in detections:
            draw_box(frame, det["bbox"], det["label"])

        out.write(frame)
        if view:
            cv2.imshow("Mask Detection", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    out.release()
    csvf.close()
    cv2.destroyAllWindows()
    print(f"Video saved to: {output_video}")
    print(f"CSV stats saved to: {csv_file}")

In [ ]:
!python -m mask_detection.mask_detector --source "videos/sample1.mp4" --out "outputs/mask_out.mp4" --view

In [ ]:
import os
import cv2
import math
import csv
from ultralytics import YOLO
import argparse

In [ ]:
source = "videos/sample3.mp4"
output_path = "outputs/distance_out.mp4"
min_distance_px = 150
use_cm = False
view = True
csv_path = "outputs/distances.csv"

In [ ]:
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def draw_box(frame, x1, y1, x2, y2, color=(0,255,0)):
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

def draw_line(frame, pt1, pt2, color=(0,0,255), text=None):
    cv2.line(frame, pt1, pt2, color, 2)
    if text:
        mid = ((pt1[0]+pt2[0])//2, (pt1[1]+pt2[1])//2)
        cv2.putText(frame, text, mid, cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

def get_center(box):
    x1, y1, x2, y2 = box
    return (int((x1+x2)/2), int((y1+y2)/2))

In [ ]:
def check_violations(boxes, min_distance_px=150, use_cm=False):
    violations = set()
    n = len(boxes)
    centers = [get_center(b) for b in boxes]
    heights = [abs(b[3]-b[1]) for b in boxes]
    closest_pairs = []

    for i in range(n):
        min_dist = float('inf')
        closest_j = -1
        for j in range(n):
            if i == j:
                continue
            dx = centers[i][0] - centers[j][0]
            dy = centers[i][1] - centers[j][1]
            dist_px = math.sqrt(dx*dx + dy*dy)

            dist_display = dist_px

            if use_cm:
                h_avg_px = (heights[i] + heights[j]) / 2
                if h_avg_px > 0:
                    px_per_cm = h_avg_px / 170.0  
                    dist_display = dist_px / px_per_cm

            if dist_display < min_dist:
                min_dist = dist_display
                closest_j = j

        if closest_j >= 0:
            violation = min_dist < min_distance_px
            if violation:
                violations.add(i)
                violations.add(closest_j)
            closest_pairs.append((i, closest_j, min_dist, min_distance_px))

    return closest_pairs, violations

In [ ]:
def run_video(source, output_path, min_distance_px=150, use_cm=False, view=False, csv_path=None):
    person_model = YOLO("models/yolov8m.pt")
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {source}")
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

    ensure_dir(os.path.dirname(output_path) or ".")
    ensure_dir(os.path.dirname(csv_path) or ".")

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width,height))

    if csv_path:
        csv_file = open(csv_path, mode='w', newline='')
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["frame", "person_id", "closest_person_id", "distance", "threshold", "violation"])

    frame_id = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_id += 1

        results = person_model(frame)[0]
        boxes = []
        for box, cls in zip(results.boxes.xyxy.cpu().numpy(), results.boxes.cls.cpu().numpy()):
            if int(cls)==0:
                boxes.append([int(box[0]), int(box[1]), int(box[2]), int(box[3])])

        closest_pairs, violations = check_violations(boxes, min_distance_px, use_cm)

        for idx, b in enumerate(boxes):
            color = (0,0,255) if idx in violations else (0,255,0)
            draw_box(frame, *b, color=color)

        for i, j, dist, threshold in closest_pairs:
            color = (0,255,0) if dist >= threshold else (0,0,255)
            text = f"{dist:.0f}" + ("cm" if use_cm else "px")
            draw_line(frame, get_center(boxes[i]), get_center(boxes[j]), color, text)
            if csv_path:
                violation = "Yes" if dist < threshold else "No"
                csv_writer.writerow([frame_id, i, j, round(dist,1), round(threshold,1), violation])

        writer.write(frame)
        if view:
            cv2.imshow("Social Distance", frame)
            if cv2.waitKey(1) & 0xFF==ord('q'):
                break

    cap.release()
    writer.release()
    if csv_path:
        csv_file.close()
    if view:
        cv2.destroyAllWindows()

    print(f"Video saved to: {output_path}")
    if csv_path:
        print(f"CSV saved to: {csv_path}")

In [ ]:
!python social_distance/distance_estimator.py --source "videos/sample2.mp4" --out "outputs/distance_out.mp4" --view

In [ ]:
import cv2
from ultralytics import YOLO
import numpy as np
import os
import csv

In [ ]:
MODEL_DIR = "models"

PERSON_MODEL = os.path.join(MODEL_DIR, "yolov8n_person.pt")
FACE_MODEL = os.path.join(MODEL_DIR, "face_yolov8n.pt")
MASK_MODEL = "runs/detect/train_new50/weights/last.pt"

DISTANCE_THRESHOLD_CM = 150  

# تحميل الموديلات
person_model = YOLO(PERSON_MODEL)
face_model = YOLO(FACE_MODEL)
mask_model = YOLO(MASK_MODEL)

# قائمة الفيديوهات
VIDEOS = [
    "videos/test1.mp4",
    "videos/test2.mp4",
    "videos/test3.mp4"
]

REAL_DIMENSIONS = [(400, 300), (400, 300), (400, 300)]

In [ ]:
PTS_SRC_LIST = [
    np.array([[23, 417], [760, 411], [446, 210], [216, 212]], dtype=np.float32),  # v1
    np.array([[4, 418], [765, 408], [413, 159], [301, 156]], dtype=np.float32),   # v2
    np.array([[10, 425], [760, 421], [486, 227], [266, 225]], dtype=np.float32)   # v3
]

In [ ]:
def box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-6)
    return iou

In [ ]:
SUMMARY_CSV = "outputs/summary.csv"

with open(SUMMARY_CSV, mode='w', newline='', encoding='utf-8') as summary_file:
    summary_writer = csv.writer(summary_file)
    summary_writer.writerow(["Video", "Distance Violations", "Mask Violations", "Protocol Violators"])

    for idx, VIDEO_INPUT in enumerate(VIDEOS):
        VIDEO_NAME = os.path.basename(VIDEO_INPUT)
        VIDEO_OUTPUT = f"outputs/final_{VIDEO_NAME}"
        CSV_OUTPUT = VIDEO_OUTPUT.replace(".mp4", ".csv")

        cap = cv2.VideoCapture(VIDEO_INPUT)
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        out = cv2.VideoWriter(VIDEO_OUTPUT, fourcc, fps, (width, height))

        csv_file = open(CSV_OUTPUT, mode='w', newline='', encoding='utf-8')
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow(["Frame", "Person_ID", "Center_X", "Center_Y", "Nearest_Person", "Distance_cm"])

        pts_src = PTS_SRC_LIST[idx]
        real_width_cm, real_height_cm = REAL_DIMENSIONS[idx]
        pixel_width = max(pts_src[:,0]) - min(pts_src[:,0])
        pixel_height = max(pts_src[:,1]) - min(pts_src[:,1])
        scale_x = real_width_cm / pixel_width
        scale_y = real_height_cm / pixel_height
        scale = (scale_x + scale_y) / 2

        frame_idx = 0
        distance_violators = set()
        mask_violators = set()

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            person_results = person_model(frame)[0]
            person_boxes = [box.cpu().numpy() for i, box in enumerate(person_results.boxes.xyxy) if int(person_results.boxes.cls[i])==0]

            face_results = face_model(frame)[0]
            face_boxes = [box.cpu().numpy() for box in face_results.boxes.xyxy]

            mask_results = mask_model(frame)[0]
            mask_boxes = [box.cpu().numpy() for box in mask_results.boxes.xyxy]
            mask_status = [int(cls) for cls in mask_results.boxes.cls]

            for face_id, face_box in enumerate(face_boxes):
                x1, y1, x2, y2 = map(int, face_box)
                is_no_mask = False
                for i, mask_box in enumerate(mask_boxes):
                    if box_iou(face_box, mask_box) > 0.2 and mask_status[i] == 1:
                        is_no_mask = True
                        mask_violators.add(face_id)
                        break
                color = (0,0,255) if is_no_mask else (0,255,0)
                label = "No Mask" if is_no_mask else "Mask"
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y2+15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

            centers = []
            for box in person_boxes:
                x1, y1, x2, y2 = box
                cx = (x1 + x2)/2
                cy = (y1 + y2)/2
                centers.append((cx, cy))

            for i, center_i in enumerate(centers):
                min_dist = float('inf')
                nearest_j = -1
                for j, center_j in enumerate(centers):
                    if i == j:
                        continue
                    dx = center_i[0] - center_j[0]
                    dy = center_i[1] - center_j[1]
                    dist_pixel = np.sqrt(dx**2 + dy**2)
                    dist_cm = dist_pixel * scale
                    if dist_cm < min_dist:
                        min_dist = dist_cm
                        nearest_j = j

                if nearest_j != -1:
                    if min_dist < DISTANCE_THRESHOLD_CM:
                        distance_violators.add(i)

                    x1c = int((person_boxes[i][0]+person_boxes[i][2])/2)
                    y1c = int((person_boxes[i][1]+person_boxes[i][3])/2)
                    x2c = int((person_boxes[nearest_j][0]+person_boxes[nearest_j][2])/2)
                    y2c = int((person_boxes[nearest_j][1]+person_boxes[nearest_j][3])/2)

                    color = (0,255,0) if min_dist >= DISTANCE_THRESHOLD_CM else (0,0,255)
                    cv2.line(frame, (x1c,y1c), (x2c,y2c), color, 2)
                    cv2.putText(frame, f"{int(min_dist)} cm", ((x1c+x2c)//2, (y1c+y2c)//2),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

                    csv_writer.writerow([frame_idx, i, int(center_i[0]), int(center_i[1]), nearest_j, int(min_dist)])

            cv2.imshow("Mask & Distance Detection", frame)
            out.write(frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        protocol_violators = distance_violators.union(mask_violators)
        summary_writer.writerow([VIDEO_NAME, len(distance_violators), len(mask_violators), len(protocol_violators)])
        print(f"{VIDEO_NAME}: Distance={len(distance_violators)}, Mask={len(mask_violators)}, Protocol={len(protocol_violators)}")

        cap.release()
        out.release()
        csv_file.close()

cv2.destroyAllWindows()

In [ ]:
!python multi_model_mask_distance.py